In [2]:
import numpy as np 
import ast
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings
warnings.filterwarnings("ignore")
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import string
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv("E:\\Project\\Recommender System\\movies_metadata.csv")
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [4]:
df.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [5]:
# remove all duplicate rows:
df = df.drop_duplicates().reset_index(drop = True)
df.duplicated().sum()

np.int64(0)

In [6]:
df = df[['title', 'overview', 'genres', 'tagline', 'vote_average', 'popularity']]
df.isnull().sum()

title               6
overview          954
genres              0
tagline         25045
vote_average        6
popularity          5
dtype: int64

In [7]:
# remove the blank titles:
df = df.dropna(subset=['title'])

# fill blank overview fill blank space:
df['overview'] = df['overview'].fillna(' ')

# df.iloc[0]['genres']      => "[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"
df['genres'] = df['genres'].apply(lambda x:" ".join([i['name'] for i in ast.literal_eval(x)]))

# df['tagline'].isnull().sum()      => 25000+ are null
df['tagline'] = df['tagline'].fillna(' ')

# details:
df.isnull().sum()

title           0
overview        0
genres          0
tagline         0
vote_average    0
popularity      0
dtype: int64

In [8]:
# combine overview, genres, and tagline to a single column as tags:
df['tags'] = df['overview'] + " " + df['genres'] + " " + df['tagline']
a = df['tags'][45]
print(a)

Soon-to-be-wed graduate student Finn Dodd develops cold feet when she suspects her fiancé is cheating on her. In order to clear her head, Finn visits her grandmother, Hy, and great aunt, Glady Joe Cleary, in Grasse, Calif. There, Finn learns that Hy and Glady Joe are members of a group of passionate quilters, and over the course of her visit she is regaled with tales of love and life by women who have collected rich experiences and much wisdom. Drama Romance There's beauty in the patterns of life.


In [9]:
# NLP task:
# remove punctuations, stop (when, and, that,....) words:
# limitazation (ending -> end ie convert the words to their root words) and then applied 
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abdul\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\abdul\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [10]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [11]:
def preprocess_text(text):
    # lower case:
    text = str(text).lower()
    
    # remove punctuation:
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # tokenize words:
    words = text.split()
    
    # remove stop words (Fixed the loop variable here):
    words = [word for word in words if word not in stop_words]
    
    # lemmatize:
    words = [lemmatizer.lemmatize(word) for word in words]

    # reassemble:
    return " ".join(words)

In [12]:
df['tags'] = df['tags'].apply(preprocess_text)

In [13]:
# reset the index
df = df.reset_index(drop = True)

In [14]:
# Create a reverse mapping of titles to DataFrame indices, removing duplicate titles:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
indices

title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Subdue                         45442
Century of Birthing            45443
Betrayal                       45444
Satan Triumphant               45445
Queerama                       45446
Length: 45447, dtype: int64

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
# apply TFIDF
tfidf = TfidfVectorizer(max_df=50000, ngram_range=(1,2), stop_words='english')

In [16]:
tfidf_matrix = tfidf.fit_transform(df['tags'])
tfidf_matrix


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2711647 stored elements and shape (45447, 1068907)>

In [17]:
def recommend(title, n=10):    # for 10 movies recommendation
    if title not in indices:
        return ["Movie not fount"]
    idx = indices[title]
    similarity_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_index = similarity_score.argsort()[::-1][1:n+1]
    return df['title'].iloc[similar_index]

In [18]:
recommend('Toy Story')

2996                         Toy Story 2
15344                        Toy Story 3
28972                    Superstar Goofy
24512                          Small Fry
6434              What's Up, Tiger Lily?
45063                      Tatu and Patu
37282    Father of Four: Never Gives Up!
31493                          Bar Sport
17184                          Group Sex
1071               Rebel Without a Cause
Name: title, dtype: object

In [21]:
import pickle
pickle.dump(tfidf_matrix, open('tfidf_matrix.pkl', 'wb'))
pickle.dump(indices, open('indices.pkl', 'wb'))
df.to_pickle('df.pkl')
pickle.dump(tfidf, open('tfidf.pkl', 'wb'))